In [1]:
import pandas as pd

In [2]:
#LOAD THE DAT SET
file_path = "ECO_PULSE_Cleaned_Data.xlsx"

df = pd.read_excel(file_path)

print("========== ECO PULSE DATASET ==========")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column Names:", df.columns.tolist())

========== ECO PULSE DATASET ==========
Rows: 2000
Columns: 32
Column Names: ['Person_ID', 'Age', 'Gender', 'Employment_Status', 'Urban_Rural', 'Outdoor_Exposure_Hours_Day', 'Sleep_Hours', 'Station_City', 'State', 'station_code', 'pin_code', 'latitude', 'longitude', 'elevation_m', 'topo_complexity', 'coastal_proximity', 'valley_factor', 'Weather_City', 'Avg_Temp_Max_C', 'Avg_Temp_Min_C', 'Avg_Precipitation_mm', 'Avg_Rain_mm', 'Avg_Wind_kmh', 'Avg_Wind_Gust_kmh', 'Dengue_Cases_2019_2024', 'Dengue_Deaths_2019_2024', 'Dengue_Cases_2024', 'Hospital_Count', 'Avg_Hospital_Density', 'Avg_Hospital_Rating', 'Total_Hospital_Reviews', 'Environmental_Dengue_Risk_Index']


In [3]:
#DATA CLEANING
df.columns = df.columns.str.strip()

before = len(df)
df = df.drop_duplicates()
removed = before - len(df)

numeric_columns = [
    "Age",
    "Outdoor_Exposure_Hours_Day",
    "Sleep_Hours",
    "Avg_Temp_Max_C",
    "Avg_Temp_Min_C",
    "Avg_Precipitation_mm",
    "Avg_Rain_mm",
    "Avg_Wind_kmh",
    "Avg_Wind_Gust_kmh",
    "Dengue_Cases_2019_2024",
    "Dengue_Deaths_2019_2024",
    "Dengue_Cases_2024",
    "Hospital_Count",
    "Avg_Hospital_Density",
    "Avg_Hospital_Rating",
    "Total_Hospital_Reviews",
    "Environmental_Dengue_Risk_Index"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Duplicate rows removed:", removed)
print("Total missing values:", df.isnull().sum().sum())

Duplicate rows removed: 0
Total missing values: 0


In [4]:
#STATISTICS
print("========== BASIC STATISTICS ==========")
print(df[numeric_columns].describe().round(2))

========== BASIC STATISTICS ==========
           Age  Outdoor_Exposure_Hours_Day  Sleep_Hours  Avg_Temp_Max_C  \
count  2000.00                     2000.00      2000.00         2000.00   
mean     48.99                        3.51         6.96           30.74   
std      18.28                        1.71         1.02            0.79   
min      18.00                        0.20         4.00           28.32   
25%      33.00                        2.40         6.30           30.64   
50%      49.00                        3.50         6.90           30.66   
75%      65.00                        4.62         7.70           30.77   
max      80.00                       10.00        10.00           33.15   

       Avg_Temp_Min_C  Avg_Precipitation_mm  Avg_Rain_mm  Avg_Wind_kmh  \
count         2000.00               2000.00      2000.00       2000.00   
mean            21.66                  3.11         3.11         16.07   
std              1.69                  1.22         1.22       

In [5]:
#GEMDER ANALYSIS
gender_analysis = (
    df["Gender"]
    .value_counts()
    .rename_axis("Gender")
    .reset_index(name="Count")
)

print("========== GENDER DISTRIBUTION ==========")
print(gender_analysis)

========== GENDER DISTRIBUTION ==========
   Gender  Count
0    Male    977
1  Female    973
2   Other     50


In [6]:
#STATE-WISE-DEBGUE-ANALYSIS
state_analysis = (
    df.groupby("State")
    .agg(
        Average_Dengue_Cases=("Dengue_Cases_2019_2024", "mean"),
        Total_Dengue_Cases=("Dengue_Cases_2019_2024", "sum"),
        Average_Dengue_Deaths=("Dengue_Deaths_2019_2024", "mean"),
        Average_Risk_Index=("Environmental_Dengue_Risk_Index", "mean"),
        Average_Hospital_Density=("Avg_Hospital_Density", "mean")
    )
    .sort_values("Average_Dengue_Cases", ascending=False)
)

print("========== STATE-WISE DENGUE ANALYSIS ==========")
print(state_analysis.round(2))

highest_state = state_analysis["Average_Dengue_Cases"].idxmax()
print("State with highest average dengue cases:", highest_state)

========== STATE-WISE DENGUE ANALYSIS ==========
                Average_Dengue_Cases  Total_Dengue_Cases  \
State                                                      
West Bengal*                111825.0             8051400   
Uttar Pradesh                99586.0            13444110   
Punjab                       66992.0             1406832   
Rajasthan                    64911.0             7724409   
Karnataka                    63367.0             6526801   
Maharashtra                  61768.0            18715704   
Delhi                        46865.0             8013915   
Gujarat                      45563.0             3098284   
Kerala                       42275.0             2790150   
Bihar                        42041.0             2564501   
Telangana                    40705.0             2035250   
Tamil Nadu                   37305.0             5558445   
Odisha                       32032.0             1057056   
Madhya Pradesh               31626.0             26

In [7]:
#Age WISE dengue ANALYSIS
age_analysis = (
    df.groupby("Age")["Dengue_Cases_2019_2024"]
    .mean()
    .reset_index()
)

age_analysis.columns = ["Age", "Average_Dengue_Cases"]

print("========== AGE-WISE DENGUE ANALYSIS ==========")
print(age_analysis.round(2).head(20))

========== AGE-WISE DENGUE ANALYSIS ==========
    Age  Average_Dengue_Cases
0    18              44920.00
1    19              40469.14
2    20              47629.55
3    21              47459.32
4    22              52969.57
5    23              44067.33
6    24              51309.63
7    25              43898.08
8    26              47435.76
9    27              48397.44
10   28              42201.88
11   29              38980.88
12   30              50891.38
13   31              51516.50
14   32              45419.00
15   33              47573.98
16   34              48162.38
17   35              44168.26
18   36              58265.78
19   37              56899.80


In [8]:
#AGE-GROUP-ANALYSIS
def age_group(age):
    if age < 18:
        return "Under 18"
    elif age <= 30:
        return "18-30"
    elif age <= 45:
        return "31-45"
    elif age <= 60:
        return "46-60"
    else:
        return "61+"

df["Age_Group"] = df["Age"].apply(age_group)

age_group_analysis = (
    df.groupby("Age_Group")
    .agg(
        People=("Person_ID", "count"),
        Average_Dengue_Cases=("Dengue_Cases_2019_2024", "mean"),
        Average_Risk=("Environmental_Dengue_Risk_Index", "mean")
    )
)

print("========== AGE GROUP ANALYSIS ==========")
print(age_group_analysis.round(2))

========== AGE GROUP ANALYSIS ==========
           People  Average_Dengue_Cases  Average_Risk
Age_Group                                            
18-30         429              46034.40         31.63
31-45         447              48863.71         31.13
46-60         486              46932.76         30.81
61+           638              47003.14         31.19


In [9]:
#ENVIRONMENTAL Average_Risk
risk_analysis = (
    df.groupby("Environmental_Dengue_Risk_Index")
    ["Dengue_Cases_2019_2024"]
    .mean()
    .reset_index()
)

risk_analysis.columns = [
    "Environmental_Risk",
    "Average_Dengue_Cases"
]

print("========== ENVIRONMENTAL RISK ANALYSIS ==========")
print(risk_analysis.round(2))

risk_correlation = df[
    ["Environmental_Dengue_Risk_Index", "Dengue_Cases_2019_2024"]
].corr().iloc[0, 1]

print(
    "Correlation: Environmental Risk vs Dengue Cases:",
    round(risk_correlation, 3)
)

========== ENVIRONMENTAL RISK ANALYSIS ==========
    Environmental_Risk  Average_Dengue_Cases
0                 14.4              46865.00
1                 17.4              18093.00
2                 17.5              42041.00
3                 17.9              28946.24
4                 18.5              54412.52
5                 19.2              10636.00
6                 19.6              32032.00
7                 20.2              99586.00
8                 22.4              31626.00
9                 23.8              40705.00
10                29.9              25651.00
11                31.0             111825.00
12                31.8              45563.00
13                47.3              63367.00
14                55.6              61768.00
15                56.3              37305.00
16                72.4              42275.00
Correlation: Environmental Risk vs Dengue Cases: 0.223


In [10]:
#WHEATHER-ANALYSIS
weather_columns = [
    "Avg_Temp_Max_C",
    "Avg_Temp_Min_C",
    "Avg_Precipitation_mm",
    "Avg_Rain_mm",
    "Avg_Wind_kmh",
    "Avg_Wind_Gust_kmh",
    "Dengue_Cases_2019_2024"
]

weather_correlation = df[weather_columns].corr()

print("========== WEATHER CORRELATION ==========")
print(weather_correlation.round(2))

========== WEATHER CORRELATION ==========
                        Avg_Temp_Max_C  Avg_Temp_Min_C  Avg_Precipitation_mm  \
Avg_Temp_Max_C                    1.00            0.42                 -0.12   
Avg_Temp_Min_C                    0.42            1.00                  0.65   
Avg_Precipitation_mm             -0.12            0.65                  1.00   
Avg_Rain_mm                      -0.12            0.65                  1.00   
Avg_Wind_kmh                     -0.09            0.73                  0.60   
Avg_Wind_Gust_kmh                -0.33            0.48                  0.26   
Dengue_Cases_2019_2024           -0.12           -0.21                  0.29   

                        Avg_Rain_mm  Avg_Wind_kmh  Avg_Wind_Gust_kmh  \
Avg_Temp_Max_C                -0.12         -0.09              -0.33   
Avg_Temp_Min_C                 0.65          0.73               0.48   
Avg_Precipitation_mm           1.00          0.60               0.26   
Avg_Rain_mm                  

In [11]:
#HOSPITAL ANALYSIS
hospital_analysis = (
    df.groupby("State")
    .agg(
        Hospital_Count=("Hospital_Count", "mean"),
        Hospital_Density=("Avg_Hospital_Density", "mean"),
        Hospital_Rating=("Avg_Hospital_Rating", "mean"),
        Dengue_Cases=("Dengue_Cases_2019_2024", "mean")
    )
    .sort_values("Dengue_Cases", ascending=False)
)

print("========== HOSPITAL ANALYSIS ==========")
print(hospital_analysis.round(2))

========== HOSPITAL ANALYSIS ==========
                Hospital_Count  Hospital_Density  Hospital_Rating  \
State                                                               
West Bengal*              68.0          17743.60             3.94   
Uttar Pradesh            125.0           1910.86             4.02   
Punjab                   117.0            867.77             4.11   
Rajasthan                153.0            425.46             4.13   
Karnataka                327.0           2444.85             3.96   
Maharashtra              521.0           6381.45             4.08   
Delhi                    130.0           4057.00             3.95   
Gujarat                  115.0            759.76             4.25   
Kerala                    44.0           1080.09             4.00   
Bihar                     11.0           1689.00             3.95   
Telangana                323.0          17677.56             3.99   
Tamil Nadu               125.0           1363.20             4.

In [12]:
#EXPORT
with pd.ExcelWriter("ECO_PULSE_Data_Analysis_Output.xlsx") as writer:

    df.to_excel(
        writer,
        sheet_name="Cleaned Data",
        index=False
    )

    gender_analysis.to_excel(
        writer,
        sheet_name="Gender Analysis",
        index=False
    )

    state_analysis.to_excel(
        writer,
        sheet_name="State Analysis"
    )

    age_analysis.to_excel(
        writer,
        sheet_name="Age Analysis",
        index=False
    )

    age_group_analysis.to_excel(
        writer,
        sheet_name="Age Group Analysis"
    )

    risk_analysis.to_excel(
        writer,
        sheet_name="Risk Analysis",
        index=False
    )

    weather_correlation.to_excel(
        writer,
        sheet_name="Weather Correlation"
    )

    hospital_analysis.to_excel(
        writer,
        sheet_name="Hospital Analysis"
    )

print("ECO PULSE DATA ANALYSIS COMPLETED SUCCESSFULLY")
print("Output file: ECO_PULSE_Data_Analysis_Output.xlsx")

ECO PULSE DATA ANALYSIS COMPLETED SUCCESSFULLY
Output file: ECO_PULSE_Data_Analysis_Output.xlsx
